In [1]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

import js
import jsGrammarOls as jsgo

import _syntax_score_helper_function as ga



In [2]:
# load syntax features
fn='F05-Syntax-Features/6-real-and-imputed-motifs.pd.df.pickle'
mDF=pd.read_pickle(fn)
mDF.head()


,phrase-length,func-odds-ratio,real-data,hyp-func-p-bonf,num-total-seqs,num-active-ens,obs-or-imputed
phrase-key,,,,,,,
ord_ori_spc=g_21_e_24_E,3.0,11.206427,"(63, 34, 7)",3.246130e-29,104.0,63.0,obs
ord_ori_spc=g_13_e_24_E,3.0,8.385350,"(108, 73, 21)",3.208155e-43,202.0,108.0,obs
ord_ori_spc=g_36_g_9_E,3.0,7.889255,"(162, 108, 42)",8.148688e-63,312.0,162.0,obs
ord_ori_spc=g_15_E_22_E,3.0,7.098879,"(107, 85, 25)",2.072536e-38,217.0,107.0,obs
ord_ori_spc=e_24_E_8_G,3.0,7.029848,"(105, 80, 29)",2.031084e-37,214.0,105.0,obs


In [3]:
sigMotifSet=set(mDF.index)
list(sigMotifSet)[:3]

['ord_ori_spc=g_8_g_28_g', 'ord_ori_spc=g_15_E_23_g', 'ord_ori_spc=g_8_E_37_G']

In [4]:
# get activity of all variants
fn='01-Calculate-Enhancer-Activity-Levels/02-generated-tables//endf.rescaledPctl_45_85.wtcollapsed.sumRnaDivSumDna-repSum.pd.df.pickle'

bDF=pd.read_pickle(fn).set_index('wt-enhancer-seq')
bDF.head()

,enhancer-id,enhancer-sites,wt-pool-sumR/sumD-sum-filtNa-Dnotnull-minActFilt0.5-log2-rescaled,abl-enhancer-seq,abl-pool-sumR/sumD-sum-filtNa-Dnotnull-minActFilt0.5-log2-rescaled,act-change-bin,enhancer-dependent-gata-ets
wt-enhancer-seq,,,,,,,
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,Otx-a WT,L1-G1R-L2-E1F-L3-E2F-L4-G2R-L5-G3F-L6,1.197798,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,0.848065,"(1, 0)",0
CATCTGAAGCTCGAGATAACTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,ORL351560,L1-G1F-L2-E1F-L3-E2F-L4-G2R-L5-G3F-L6,0.856235,CATCTGAAGCTCGAAATAACTAACGCAAGTTTTCGAAAAGCAAATT...,-1.100699,"(0, -1)",0
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAGATATTAAGATAGGA,ORL108031,L1-G1R-L2-E1F-L3-E2F-L4-G2F-L5-G3F-L6,0.777169,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,0.706492,"(0, 0)",0
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTCCTATCTTA,ORL414244,L1-G1R-L2-E1F-L3-E2F-L4-G2R-L5-G3R-L6,NaN,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,-1.486866,"(0, -1)",0
CATCTGAAGCTCGTTATCTCTAACTTCCGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,ORL280162,L1-G1R-L2-E1R-L3-E2F-L4-G2R-L5-G3F-L6,0.518839,CATCTGAAGCTCGTTATTTCTAACTTGCGTTTTCGAAAAGCAAATT...,-1.322202,"(0, -1)",0


In [5]:
len(bDF)

460800

In [6]:
# Get which variants were dependent on gata/ets
geDF=bDF.loc[bDF['enhancer-dependent-gata-ets']==1,:].copy(deep=True)

seqs_dependent_gataets=set(geDF.index)
print(len(geDF))
geDF.head()


52139


,enhancer-id,enhancer-sites,wt-pool-sumR/sumD-sum-filtNa-Dnotnull-minActFilt0.5-log2-rescaled,abl-enhancer-seq,abl-pool-sumR/sumD-sum-filtNa-Dnotnull-minActFilt0.5-log2-rescaled,act-change-bin,enhancer-dependent-gata-ets
wt-enhancer-seq,,,,,,,
CATCTGAAGCTCGAGATAACTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTCCTATCTTA,ORL373222,L1-G1F-L2-E1F-L3-E2F-L4-G2R-L5-G3R-L6,1.249433,CATCTGAAGCTCGAAATAACTAACGCAAGTTTTCGAAAAGCAAATT...,-0.258002,"(1, -1)",1
CATCTGAAGCTCGAGATAACTAACTTCCGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,ORL317857,L1-G1F-L2-E1R-L3-E2F-L4-G2R-L5-G3F-L6,1.375287,CATCTGAAGCTCGAAATAACTAACTTGCGTTTTCGAAAAGCAAATT...,NaN,"(1, -1)",1
CATCTGAAGCTCGTTATCTCTAACTTCCGTTTTCGAAAAGGAAATTGCTCAGATATTCCTATCTTA,ORL383871,L1-G1R-L2-E1R-L3-E2F-L4-G2F-L5-G3R-L6,1.614942,CATCTGAAGCTCGTTATTTCTAACTTGCGTTTTCGAAAAGCAAATT...,-0.449367,"(1, -1)",1
CATCTGAAGCTCGAGATAACTAACTTCCGTTTTCGAAAAGGAAATTGCTCAGATATTCCTATCTTA,ORL190541,L1-G1F-L2-E1R-L3-E2F-L4-G2F-L5-G3R-L6,1.343622,CATCTGAAGCTCGAAATAACTAACTTGCGTTTTCGAAAAGCAAATT...,NaN,"(1, -1)",1
CATCTGAAGCTCGAGATAACTAACTTCCGTTTTCGAAATTTCCTTTGCTCAATATCTCCTATCTTA,ORL200984,L1-G1F-L2-E1R-L3-E2R-L4-G2R-L5-G3R-L6,1.336315,CATCTGAAGCTCGAAATAACTAACTTGCGTTTTCGAAATTTGCTTT...,NaN,"(1, -1)",1


In [7]:
annoCol='activity-label'

f_cutoff=1
n_cutoff=0

annoList=[]

for en in bDF.index:

    act=bDF.at[en,'wt-pool-sumR/sumD-sum-filtNa-Dnotnull-minActFilt0.5-log2-rescaled']

    # if variants is functional AND is driven by gata/ets
    if (act >= f_cutoff) and (en in seqs_dependent_gataets):
        annoList.append(1)

    elif act <= n_cutoff:
        annoList.append(0)

    else:
        annoList.append(np.NaN)

bDF[annoCol]=annoList

bDF[annoCol].value_counts()

activity-label
0.0    143799
1.0     52139
Name: count, dtype: int64

In [8]:

# the dataframe to be inputted to calculate syntax scores
featDF=bDF
featDF.head()

,enhancer-id,enhancer-sites,wt-pool-sumR/sumD-sum-filtNa-Dnotnull-minActFilt0.5-log2-rescaled,abl-enhancer-seq,abl-pool-sumR/sumD-sum-filtNa-Dnotnull-minActFilt0.5-log2-rescaled,act-change-bin,enhancer-dependent-gata-ets,activity-label
wt-enhancer-seq,,,,,,,,
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,Otx-a WT,L1-G1R-L2-E1F-L3-E2F-L4-G2R-L5-G3F-L6,1.197798,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,0.848065,"(1, 0)",0,NaN
CATCTGAAGCTCGAGATAACTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,ORL351560,L1-G1F-L2-E1F-L3-E2F-L4-G2R-L5-G3F-L6,0.856235,CATCTGAAGCTCGAAATAACTAACGCAAGTTTTCGAAAAGCAAATT...,-1.100699,"(0, -1)",0,NaN
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAGATATTAAGATAGGA,ORL108031,L1-G1R-L2-E1F-L3-E2F-L4-G2F-L5-G3F-L6,0.777169,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,0.706492,"(0, 0)",0,NaN
CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTCCTATCTTA,ORL414244,L1-G1R-L2-E1F-L3-E2F-L4-G2R-L5-G3R-L6,NaN,CATCTGAAGCTCGTTATTTCTAACGCAAGTTTTCGAAAAGCAAATT...,-1.486866,"(0, -1)",0,NaN
CATCTGAAGCTCGTTATCTCTAACTTCCGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA,ORL280162,L1-G1R-L2-E1R-L3-E2F-L4-G2R-L5-G3F-L6,0.518839,CATCTGAAGCTCGTTATTTCTAACTTGCGTTTTCGAAAAGCAAATT...,-1.322202,"(0, -1)",0,NaN


In [10]:
outdir=f'F05i-Syntax-Scores/'
print(outdir)

F05i-Syntax-Scores/


In [12]:

effectSize='odds-ratio'

print(effectSize)

# If writing
eDF=ga.get_all_grammar_motifs_BATCH(featDF.index,
                                    sigMotifSet,
                                    mDF,
                                    f'func-{effectSize}',
                                    f'',
                                    skip_aff=True,
                                    activatorOnly=True)

eDF.to_pickle(f'{outdir}/1i-all-ens-with-motifs-annotated.funcAnnotationCol={annoCol}.effectSize={effectSize}.pandas.dataframe.pickle')

# Collapse onto sequences
Col2Func={'fc':list}

eDF=eDF.loc[:,['seq','type']+list(Col2Func.keys())].groupby(['seq','type']).agg(Col2Func).reset_index().set_index('seq')

eDF.columns=['Type','M']

aDF=eDF.loc[eDF.Type=='F',['M']]
aDF.columns=['Ma']

finalDF=pd.concat([aDF],axis=1)

# finalDF['functional']=[En2Func[en] for en in finalDF.index]

# Sum features

for colBase in ['M']:
    for Type in ['a']:
        col=colBase+Type

        newcol=col+'-sum'
        finalDF[newcol]=finalDF[col].apply(lambda l: sum(l) if type(l)==list else np.NaN)


finalDF.to_pickle(f'{outdir}/2i-all-seqs-with-motif-info.funcAnnotationCol={annoCol}.effectSize={effectSize}.pandas.dataframe.pickle')

finalDF['Ma-sum-nafill1']=finalDF['Ma-sum'].fillna(1)

finalDF.to_pickle(f'{outdir}/3i-all-seqs.fillNaN.funcAnnotationCol={annoCol}.effectSize={effectSize}.pandas.dataframe.pickle')
    
print('done')

odds-ratio
done
